# 구조환경지표 28개 보간 전 통합패널

GitHub 이슈 #80의 승인된 입력 8개만 명시적 화이트리스트로 읽어 2016–2024년 × 전국·17개 시도 × 28개 지표의 완전 격자를 생성한다. 입력값은 재계산하거나 변환하지 않으며, 전국값을 시도 평균으로 만들지 않는다. 보간·결측 대체·정규화·표준화·방향 변환·가중치·구조환경지수 산출은 수행하지 않는다.

## 경로

- 입력 루트: `data/interim/`
- 매니페스트: `configs/structural_indicators_verification.yaml`
- 통합패널: `data/processed/구조환경지표_28개_보간전_기준패널.csv`
- 입력 파일 QA: `reports/20260804_구조환경지표_28개_입력파일_QA.csv`
- 패널 완전성·관측상태 QA: `reports/20260804_구조환경지표_28개_패널완전성_관측상태_QA.csv`
- raw_sources 출처 계보 QA: `reports/20260804_구조환경지표_28개_raw_sources_출처계보_QA.csv`
- 입력값 보존 QA: `reports/20260804_구조환경지표_28개_입력값보존_QA.csv`
- QA 요약: `reports/20260804_구조환경지표_28개_보간전_통합패널_QA.md`

In [1]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'configs' / 'structural_indicators_verification.yaml').exists():
            return candidate
    raise FileNotFoundError('저장소 루트를 찾지 못했습니다.')

ROOT = find_repo_root(Path.cwd())
INPUT_ROOT = ROOT / 'data' / 'interim'
MANIFEST_PATH = ROOT / 'configs' / 'structural_indicators_verification.yaml'
PANEL_PATH = ROOT / 'data' / 'processed' / '구조환경지표_28개_보간전_기준패널.csv'
INPUT_QA_PATH = ROOT / 'reports' / '20260804_구조환경지표_28개_입력파일_QA.csv'
PANEL_QA_PATH = ROOT / 'reports' / '20260804_구조환경지표_28개_패널완전성_관측상태_QA.csv'
SOURCE_QA_PATH = ROOT / 'reports' / '20260804_구조환경지표_28개_raw_sources_출처계보_QA.csv'
VALUE_QA_PATH = ROOT / 'reports' / '20260804_구조환경지표_28개_입력값보존_QA.csv'
REPORT_PATH = ROOT / 'reports' / '20260804_구조환경지표_28개_보간전_통합패널_QA.md'

YEARS = list(range(2016, 2025))
YEAR_COLUMNS = [str(year) for year in YEARS]
REGIONS = ['전국', '서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종', '경기', '강원', '충북', '충남', '전북', '전남', '경북', '경남', '제주']
PROVINCES = REGIONS[1:]
VALUE_ATOL = 1e-12

LEGACY_FILE = '구조환경지표_21개_검증본.csv'
EMPLOYMENT_FILE = '2016-2024_지역별고용조사_청년층_정규직_근로자_비율_연도평균.csv'
NEW_FILE_TO_ID = OrderedDict([
    (EMPLOYMENT_FILE, 'youth_regular_employment_rate'),
    ('현재_주택가격_2016-2024.csv', 'housing_price'),
    ('전체_가구_자가점유비율_2016-2024.csv', 'household_own_occupancy_rate'),
    ('임차가구_연간_주거비_HCC_2016-2024.csv', 'renter_household_annual_housing_cost_hcc'),
    ('가족실태조사_가사노동_공동_참여도_2020_2023.csv', 'household_chore_equal_participation'),
    ('가족실태조사_돌봄노동_공동_참여도_2020_2023.csv', 'childcare_equal_participation'),
    ('가족실태조사_사회경제적_지위에_대한_인식_2020_2023.csv', 'socioeconomic_status_perception'),
])
WHITELIST = [LEGACY_FILE, *NEW_FILE_TO_ID.keys()]

LEGACY_NAME_TO_ID = OrderedDict([
    ('청년고용률', 'youth_employment_rate'),
    ('소득만족도', 'income_satisfaction'),
    ('소득수준', 'income_level'),
    ('보육시설 보급률', 'childcare_capacity_rate'),
    ('방과후 돌봄시설 보급도', 'after_school_care_supply'),
    ('사교육비 지출액', 'private_education_cost'),
    ('문화기반시설 보급도', 'cultural_facilities_supply'),
    ('도시공원 보급도', 'urban_park_supply'),
    ('여가생활 만족도', 'leisure_satisfaction'),
    ('분만실 병상수 보급도', 'delivery_bed_supply'),
    ('소아청소년과 전문인력 보급도', 'pediatric_specialist_supply'),
    ('산후조리원 보급도', 'postpartum_center_supply'),
    ('산후조리원 이용 요금', 'postpartum_center_fee'),
    ('어린이 교통사고 발생률', 'child_traffic_accident_rate'),
    ('사회 안전에 대한 인식', 'social_safety_perception'),
    ('근로시간', 'work_hours'),
    ('육아휴직 사용률', 'parental_leave_usage'),
    ('가족친화인증기업 비율', 'family_friendly_certification_rate'),
    ('결혼에 대한 인식', 'marriage_perception'),
    ('출산에 대한 인식', 'childbirth_perception'),
    ('가사 분담에 대한 성평등 인식', 'housework_gender_equality'),
])
LEGACY_NAME_DIFFERENCES = {
    '소득수준': '소득 수준',
    '분만실 병상수 보급도': '(대체)분만실 병상수 보급도',
    '소아청소년과 전문인력 보급도': '(대체)소아청소년과 전문인력 보급도',
    '가족친화인증기업 비율': '가족친화 인증기업 비율',
    '가사 분담에 대한 성평등 인식': '가사분담에 대한 성평등 인식',
}

EXPECTED_NEW_LABELS = {
    '현재_주택가격_2016-2024.csv': '현재 주택가격',
    '전체_가구_자가점유비율_2016-2024.csv': '전체 가구 자가점유비율',
    '임차가구_연간_주거비_HCC_2016-2024.csv': '임차가구 연간 주거비 HCC',
    '가족실태조사_가사노동_공동_참여도_2020_2023.csv': '가사노동 공동 참여도',
    '가족실태조사_돌봄노동_공동_참여도_2020_2023.csv': '돌봄노동 공동 참여도',
    '가족실태조사_사회경제적_지위에_대한_인식_2020_2023.csv': '사회경제적 지위에 대한 인식',
}

print('저장소:', ROOT)
print('화이트리스트 입력 8개:', len(WHITELIST))
for name in WHITELIST:
    print(' -', INPUT_ROOT / name)
print('통합패널:', PANEL_PATH)
print('QA:', INPUT_QA_PATH, PANEL_QA_PATH, SOURCE_QA_PATH, VALUE_QA_PATH, REPORT_PATH, sep='\n - ')

저장소: D:\University\yumocha\yumocha
화이트리스트 입력 8개: 8
 - D:\University\yumocha\yumocha\data\interim\구조환경지표_21개_검증본.csv
 - D:\University\yumocha\yumocha\data\interim\2016-2024_지역별고용조사_청년층_정규직_근로자_비율_연도평균.csv
 - D:\University\yumocha\yumocha\data\interim\현재_주택가격_2016-2024.csv
 - D:\University\yumocha\yumocha\data\interim\전체_가구_자가점유비율_2016-2024.csv
 - D:\University\yumocha\yumocha\data\interim\임차가구_연간_주거비_HCC_2016-2024.csv
 - D:\University\yumocha\yumocha\data\interim\가족실태조사_가사노동_공동_참여도_2020_2023.csv
 - D:\University\yumocha\yumocha\data\interim\가족실태조사_돌봄노동_공동_참여도_2020_2023.csv
 - D:\University\yumocha\yumocha\data\interim\가족실태조사_사회경제적_지위에_대한_인식_2020_2023.csv
통합패널: D:\University\yumocha\yumocha\data\processed\구조환경지표_28개_보간전_기준패널.csv
QA:
 - D:\University\yumocha\yumocha\reports\20260804_구조환경지표_28개_입력파일_QA.csv
 - D:\University\yumocha\yumocha\reports\20260804_구조환경지표_28개_패널완전성_관측상태_QA.csv
 - D:\University\yumocha\yumocha\reports\20260804_구조환경지표_28개_raw_sources_출처계보_QA.csv
 - D:\University\yumoc

## 1. 매니페스트와 단위 검증

In [2]:
with MANIFEST_PATH.open('r', encoding='utf-8') as file:
    manifest = yaml.safe_load(file)

indicators = manifest['indicators']
assert len(indicators) == 28, f"매니페스트 지표 수 오류: {len(indicators)}"
indicator_ids = [item['id'] for item in indicators]
assert len(set(indicator_ids)) == 28, '매니페스트 지표 ID가 고유하지 않습니다.'
assert all(str(item.get('unit', '')).strip() for item in indicators), '매니페스트 unit 결측이 있습니다.'
housework_meta = next(item for item in indicators if item['id'] == 'housework_gender_equality')
assert housework_meta['direction'] == 'center_or_positive', 'housework_gender_equality 방향성이 변경되었습니다.'

metadata = pd.DataFrame([
    {
        '지표_id': item['id'],
        '지표명': item['name'],
        '대분류': item['category'],
        '세부영역': item['subcategory'],
        '단위': item['unit'],
        '방향': item['direction'],
    }
    for item in indicators
])
assert not metadata.isna().any().any(), '최종 패널 필수 메타데이터에 결측이 있습니다.'

manifest_name_by_id = metadata.set_index('지표_id')['지표명'].to_dict()
assert len(LEGACY_NAME_DIFFERENCES) == 5
for input_name, manifest_name in LEGACY_NAME_DIFFERENCES.items():
    mapped_id = LEGACY_NAME_TO_ID[input_name]
    assert manifest_name_by_id[mapped_id] == manifest_name

print('매니페스트 YAML 파싱: PASS')
print('지표 수/고유 ID:', len(indicators), '/', len(set(indicator_ids)))
print('unit 결측:', sum(not str(item.get('unit', '')).strip() for item in indicators))
print('housework_gender_equality 방향:', housework_meta['direction'])
display(metadata)

매니페스트 YAML 파싱: PASS
지표 수/고유 ID: 28 / 28
unit 결측: 0
housework_gender_equality 방향: center_or_positive


,지표_id,지표명,대분류,세부영역,단위,방향
0,youth_employment_rate,청년고용률,경제·고용·주거,고용여건,%,positive
1,work_hours,근로시간,사회·문화,일·가정 양립 여건,시간/월(상용근로자 1인당),negative
2,income_satisfaction,소득만족도,경제·고용·주거,경제적 여건,점(1–5점 척도),positive
3,income_level,소득 수준,경제·고용·주거,경제적 여건,천원/인·년,positive
4,childcare_capacity_rate,보육시설 보급률,가족·생활,돌봄여건,%,positive
5,after_school_care_supply,방과후 돌봄시설 보급도,가족·생활,돌봄여건,개소/아동 1천 명,positive
6,private_education_cost,사교육비 지출액,가족·생활,돌봄여건,만원/학생 1인·월,negative
7,cultural_facilities_supply,문화기반시설 보급도,가족·생활,여가 인프라,개소/인구 10만 명,positive
8,urban_park_supply,도시공원 보급도,가족·생활,여가 인프라,천㎡/도시지역 인구 1천 명,positive
9,leisure_satisfaction,여가생활 만족도,가족·생활,여가 인프라,점(1–5점 척도),positive


## 2. raw_sources 출처 변환과 계보 보존

표시 문자열은 Unicode NFC → 양끝 공백 제거 → 연속 공백·개행 축약을 적용한다. 중복 키는 여기에 영문 casefold를 추가하고 지표 내부에서 첫 표시를 보존한다. `role`은 최종 출처에 포함하지 않는다. `housing_price`의 `comparison_series`만 QA 계보에 남기고 최종 출처에서 제외한다.

In [3]:
def nonblank(value) -> bool:
    return value is not None and bool(str(value).strip())

def normalize_source_display(value: str) -> str:
    nfc = unicodedata.normalize('NFC', str(value))
    return re.sub(r'\s+', ' ', nfc.strip())

lineage_rows = []
final_source_by_id = {}

for item in indicators:
    seen = set()
    components = []
    raw_sources = item.get('raw_sources', [])
    for order, raw_source in enumerate(raw_sources, start=1):
        if nonblank(raw_source.get('source')):
            selected_field = 'source'
            selected_raw = raw_source['source']
        elif nonblank(raw_source.get('file_name')):
            selected_field = 'file_name'
            selected_raw = raw_source['file_name']
        else:
            raise AssertionError(f"출처와 파일명이 모두 비었습니다: {item['id']} raw_sources[{order}]")

        selected_display = normalize_source_display(selected_raw)
        dedup_key = selected_display.casefold()
        role = raw_source.get('role')
        include = True
        exclusion_reason = ''
        if item['id'] == 'housing_price' and role == 'comparison_series':
            include = False
            exclusion_reason = 'QA comparison series'
        elif dedup_key in seen:
            include = False
            exclusion_reason = 'duplicate source'
        else:
            seen.add(dedup_key)
            components.append(selected_display)

        lineage_rows.append({
            '지표_id': item['id'],
            'raw_source_order': order,
            'role': role,
            'source': raw_source.get('source'),
            'file_name': raw_source.get('file_name'),
            '선택필드': selected_field,
            '선택출처': selected_display,
            '최종출처포함여부': include,
            '제외사유': exclusion_reason,
            'raw_source_json': json.dumps(raw_source, ensure_ascii=False, sort_keys=False),
        })

    final_source = '; '.join(components)
    assert final_source, f"최종 출처가 비었습니다: {item['id']}"
    final_source_by_id[item['id']] = final_source

source_lineage = pd.DataFrame(lineage_rows, columns=[
    '지표_id', 'raw_source_order', 'role', 'source', 'file_name', '선택필드', '선택출처',
    '최종출처포함여부', '제외사유', 'raw_source_json'
])
assert len(source_lineage) == 53, f"raw_sources 항목 수 오류: {len(source_lineage)}"
comparison_excluded = source_lineage['제외사유'].eq('QA comparison series').sum()
assert comparison_excluded == 1, f"comparison_series 제외 수 오류: {comparison_excluded}"
included_source_components = int(source_lineage['최종출처포함여부'].sum())
assert included_source_components == 52, f"최종 출처 구성요소 수 오류: {included_source_components}"
assert len(final_source_by_id) == 28 and all(final_source_by_id.values())

print('raw_sources 원본 항목:', len(source_lineage))
print('comparison_series 제외:', comparison_excluded)
print('최종 출처 구성요소:', included_source_components)
print('출처가 비는 지표:', sum(not value for value in final_source_by_id.values()))
display(source_lineage[source_lineage['지표_id'].eq('housing_price')])

raw_sources 원본 항목: 53
comparison_series 제외: 1
최종 출처 구성요소: 52
출처가 비는 지표: 0


,지표_id,raw_source_order,role,source,file_name,선택필드,선택출처,최종출처포함여부,제외사유,raw_source_json
45,housing_price,1,microdata,MDIS 주거실태조사 일반가구(제공) 0717 추출본,NaN,source,MDIS 주거실태조사 일반가구(제공) 0717 추출본,True,,"{""role"": ""microdata"", ""source"": ""MDIS 주거실태조사 일반가구(제공) 0717 추출본"", ""survey_name"": ""주거실태조사"", ""data_type"": ""일반가구(제공) 마이크로데이터 CSV"", ""analysis_years"": ""2016–2024년..."
46,housing_price,2,comparison_series,한국부동산원 월별 중위매매가격(주택종합),NaN,source,한국부동산원 월별 중위매매가격(주택종합),False,QA comparison series,"{""role"": ""comparison_series"", ""source"": ""한국부동산원 월별 중위매매가격(주택종합)"", ""data_type"": ""전국·17개 시도 월별 CSV"", ""analysis_years"": ""2016–2024년(108개월)"", ""file_pattern"": ""(..."


## 3. 승인된 입력 8개 로드·스키마 검증·공통 long 변환

`WHITELIST`에 적은 정확한 파일만 직접 연다. `data/interim/`의 다른 파일은 열거하거나 검사하지 않는다.

In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def as_json(value) -> str:
    return json.dumps(value, ensure_ascii=False)

def normalize_region(value: str) -> str:
    text = str(value).strip()
    return '전국' if text == '전체' else text

def missing_mask(series: pd.Series) -> pd.Series:
    stripped = series.astype('string').str.strip()
    return series.isna() | stripped.isin(['', '-'])

def numeric_without_transformation(series: pd.Series, context: str) -> pd.Series:
    source_missing = missing_mask(series)
    numeric = pd.to_numeric(series, errors='coerce')
    failures = (~source_missing) & numeric.isna()
    if failures.any():
        examples = series[failures].astype(str).drop_duplicates().head(10).tolist()
        raise AssertionError(f"숫자형 변환 규칙이 모호합니다: {context}, 값={examples}")
    return numeric.astype('float64')

def key_duplicate_count(frame: pd.DataFrame) -> int:
    return int(frame.duplicated(['지역', '연도', '지표_id'], keep=False).sum())

def make_long(frame: pd.DataFrame, file_name: str, region_col: str, year_columns, indicator_id=None) -> pd.DataFrame:
    base = frame.copy()
    base['원본행번호'] = np.arange(1, len(base) + 1)
    if indicator_id is not None:
        base['지표_id'] = indicator_id
    long = base.melt(
        id_vars=['원본행번호', region_col, '지표_id'],
        value_vars=list(year_columns),
        var_name='연도',
        value_name='원본값',
    )
    long['지역'] = long[region_col].map(normalize_region)
    long['연도'] = long['연도'].astype(int)
    long['측정값'] = numeric_without_transformation(long['원본값'], file_name)
    long['원본파일'] = file_name
    long['원본행존재'] = True
    return long[['지역', '연도', '지표_id', '측정값', '원본값', '원본파일', '원본행번호', '원본행존재']]

missing_files = [name for name in WHITELIST if not (INPUT_ROOT / name).is_file()]
assert not missing_files, f"승인된 입력 파일 누락: {missing_files}"
loaded = {name: pd.read_csv(INPUT_ROOT / name, encoding='utf-8-sig') for name in WHITELIST}
long_frames = []
input_qa_rows = []

legacy = loaded[LEGACY_FILE].copy()
expected_legacy_columns = ['지역', '대영역', '세부영역', '세부지표', *[str(y) for y in range(2015, 2026)], '검증상태']
assert legacy.columns.tolist() == expected_legacy_columns, f"21개 검증본 스키마 불일치: {legacy.columns.tolist()}"
legacy_labels = set(legacy['세부지표'].dropna().astype(str).str.strip())
assert legacy_labels == set(LEGACY_NAME_TO_ID), f"21개 지표 표현 불일치: {legacy_labels ^ set(LEGACY_NAME_TO_ID)}"
legacy['지표_id'] = legacy['세부지표'].astype(str).str.strip().map(LEGACY_NAME_TO_ID)
assert legacy['지표_id'].notna().all()
legacy_long = make_long(legacy, LEGACY_FILE, '지역', YEAR_COLUMNS)
assert set(legacy_long['지역']) == set(REGIONS)
assert key_duplicate_count(legacy_long) == 0
long_frames.append(legacy_long)

def add_input_qa(file_name: str, frame: pd.DataFrame, long: pd.DataFrame, ids) -> None:
    path = INPUT_ROOT / file_name
    input_qa_rows.append({
        '파일명': file_name,
        'SHA-256': sha256_file(path),
        '파일크기_bytes': path.stat().st_size,
        '수정시각': pd.Timestamp(path.stat().st_mtime, unit='s', tz='Asia/Seoul').isoformat(),
        '행 수': len(frame),
        '열 수': len(frame.columns),
        '열 이름': as_json(frame.columns.tolist()),
        '자료형': as_json({column: str(dtype) for column, dtype in frame.dtypes.items()}),
        '대응 지표': '; '.join(ids),
        '지역 범위': as_json([region for region in REGIONS if region in set(long['지역'])]),
        '연도 범위': f"{long['연도'].min()}–{long['연도'].max()} ({', '.join(map(str, sorted(long['연도'].unique())))})",
        '중복 키 수': key_duplicate_count(long),
    })

add_input_qa(LEGACY_FILE, loaded[LEGACY_FILE], legacy_long, list(LEGACY_NAME_TO_ID.values()))

employment = loaded[EMPLOYMENT_FILE].copy()
assert employment.columns.tolist() == ['시도', *YEAR_COLUMNS], f"고용 CSV 스키마 불일치: {employment.columns.tolist()}"
employment_regions = employment['시도'].astype(str).str.strip()
assert len(employment) == 17 and employment_regions.nunique() == 17
assert set(employment_regions) == set(PROVINCES), f"고용 CSV 17개 시도 불일치: {set(employment_regions) ^ set(PROVINCES)}"
employment_long = make_long(employment, EMPLOYMENT_FILE, '시도', YEAR_COLUMNS, 'youth_regular_employment_rate')
assert key_duplicate_count(employment_long) == 0
long_frames.append(employment_long)
add_input_qa(EMPLOYMENT_FILE, employment, employment_long, ['youth_regular_employment_rate'])

for file_name, indicator_id in list(NEW_FILE_TO_ID.items())[1:]:
    frame = loaded[file_name].copy()
    year_columns = ['2020', '2023'] if file_name.startswith('가족실태조사_') else YEAR_COLUMNS
    expected_columns = ['지역', '세부지표', *year_columns]
    assert frame.columns.tolist() == expected_columns, f"{file_name} 스키마 불일치: {frame.columns.tolist()}"
    actual_labels = frame['세부지표'].dropna().astype(str).str.strip().unique().tolist()
    assert actual_labels == [EXPECTED_NEW_LABELS[file_name]], f"{file_name} 지표 표현 불일치: {actual_labels}"
    normalized_regions = frame['지역'].map(normalize_region)
    assert len(frame) == 18 and normalized_regions.nunique() == 18
    assert set(normalized_regions) == set(REGIONS), f"{file_name} 지역 불일치: {set(normalized_regions) ^ set(REGIONS)}"
    long = make_long(frame, file_name, '지역', year_columns, indicator_id)
    assert key_duplicate_count(long) == 0
    long_frames.append(long)
    add_input_qa(file_name, frame, long, [indicator_id])

input_long = pd.concat(long_frames, ignore_index=True)
input_file_qa = pd.DataFrame(input_qa_rows)
assert len(input_file_qa) == 8
assert len(input_long) == 4140, f"원본 존재 조합 오류: {len(input_long)}"
assert key_duplicate_count(input_long) == 0, '동일 키 입력 중복이 있습니다.'
assert set(input_long['지표_id']) == set(indicator_ids), '입력과 매니페스트 지표가 1:1 대응하지 않습니다.'
assert input_long['측정값'].notna().sum() == 3266, f"숫자 관측 수 오류: {input_long['측정값'].notna().sum()}"
assert input_long['측정값'].isna().sum() == 874, f"원본 셀 결측 수 오류: {input_long['측정값'].isna().sum()}"

print('승인된 파일 로드:', len(input_file_qa))
print('원본 존재 조합:', len(input_long))
print('숫자 관측 / 원본 셀 결측:', input_long['측정값'].notna().sum(), '/', input_long['측정값'].isna().sum())
display(input_file_qa)

승인된 파일 로드: 8
원본 존재 조합: 4140
숫자 관측 / 원본 셀 결측: 3266 / 874


,파일명,SHA-256,파일크기_bytes,수정시각,행 수,열 수,열 이름,자료형,대응 지표,지역 범위,연도 범위,중복 키 수
0,구조환경지표_21개_검증본.csv,a4bca069e57f1405fed3b229ab22a6d3f503585fa415a5db956b62672c5e436e,98545,2026-07-26T03:07:05+09:00,377,16,"[""지역"", ""대영역"", ""세부영역"", ""세부지표"", ""2015"", ""2016"", ""2017"", ""2018"", ""2019"", ""2020"", ""2021"", ""2022"", ""2023"", ""2024"", ""2025"", ""검증상태""]","{""지역"": ""str"", ""대영역"": ""str"", ""세부영역"": ""str"", ""세부지표"": ""str"", ""2015"": ""float64"", ""2016"": ""str"", ""2017"": ""str"", ""2018"": ""str"", ""2019"": ""str"", ""2020"": ""float64"", ...",youth_employment_rate; income_satisfaction; income_level; childcare_capacity_rate; after_school_care_supply; private_education_cost; cultural_facilities_sup...,"[""전국"", ""서울"", ""부산"", ""대구"", ""인천"", ""광주"", ""대전"", ""울산"", ""세종"", ""경기"", ""강원"", ""충북"", ""충남"", ""전북"", ""전남"", ""경북"", ""경남"", ""제주""]","2016–2024 (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)",0
1,2016-2024_지역별고용조사_청년층_정규직_근로자_비율_연도평균.csv,64554014fc3d75dd07ec9732a821329532cb7c6c292fccd25224cff2533d5f34,953,2026-08-04T12:11:52.909379482+09:00,17,10,"[""시도"", ""2016"", ""2017"", ""2018"", ""2019"", ""2020"", ""2021"", ""2022"", ""2023"", ""2024""]","{""시도"": ""str"", ""2016"": ""float64"", ""2017"": ""float64"", ""2018"": ""float64"", ""2019"": ""float64"", ""2020"": ""float64"", ""2021"": ""float64"", ""2022"": ""float64"", ""2023"": ""...",youth_regular_employment_rate,"[""서울"", ""부산"", ""대구"", ""인천"", ""광주"", ""대전"", ""울산"", ""세종"", ""경기"", ""강원"", ""충북"", ""충남"", ""전북"", ""전남"", ""경북"", ""경남"", ""제주""]","2016–2024 (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)",0
2,현재_주택가격_2016-2024.csv,f47d0749ee5ca2b80e4ebb24f9afe065d095f0e87851d2bf71ecd8c923ced519,1860,2026-08-02T07:18:01.575999975+09:00,18,11,"[""지역"", ""세부지표"", ""2016"", ""2017"", ""2018"", ""2019"", ""2020"", ""2021"", ""2022"", ""2023"", ""2024""]","{""지역"": ""str"", ""세부지표"": ""str"", ""2016"": ""float64"", ""2017"": ""float64"", ""2018"": ""float64"", ""2019"": ""float64"", ""2020"": ""float64"", ""2021"": ""float64"", ""2022"": ""floa...",housing_price,"[""전국"", ""서울"", ""부산"", ""대구"", ""인천"", ""광주"", ""대전"", ""울산"", ""세종"", ""경기"", ""강원"", ""충북"", ""충남"", ""전북"", ""전남"", ""경북"", ""경남"", ""제주""]","2016–2024 (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)",0
3,전체_가구_자가점유비율_2016-2024.csv,1a0e66333be40b0fd18c54bcaee1527b38098e9ba2dbe2a3c36a1cc0833f9c13,1613,2026-08-02T07:18:01.575999975+09:00,18,11,"[""지역"", ""세부지표"", ""2016"", ""2017"", ""2018"", ""2019"", ""2020"", ""2021"", ""2022"", ""2023"", ""2024""]","{""지역"": ""str"", ""세부지표"": ""str"", ""2016"": ""float64"", ""2017"": ""float64"", ""2018"": ""float64"", ""2019"": ""float64"", ""2020"": ""float64"", ""2021"": ""float64"", ""2022"": ""floa...",household_own_occupancy_rate,"[""전국"", ""서울"", ""부산"", ""대구"", ""인천"", ""광주"", ""대전"", ""울산"", ""세종"", ""경기"", ""강원"", ""충북"", ""충남"", ""전북"", ""전남"", ""경북"", ""경남"", ""제주""]","2016–2024 (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)",0
4,임차가구_연간_주거비_HCC_2016-2024.csv,ddedb15c92c66bee07eff2494c69d8f118d342c0d3f5fcbe75bae892db59fa2e,1476,2026-08-02T07:18:01.575999975+09:00,18,11,"[""지역"", ""세부지표"", ""2016"", ""2017"", ""2018"", ""2019"", ""2020"", ""2021"", ""2022"", ""2023"", ""2024""]","{""지역"": ""str"", ""세부지표"": ""str"", ""2016"": ""float64"", ""2017"": ""int64"", ""2018"": ""int64"", ""2019"": ""int64"", ""2020"": ""int64"", ""2021"": ""int64"", ""2022"": ""int64"", ""2023""...",renter_household_annual_housing_cost_hcc,"[""전국"", ""서울"", ""부산"", ""대구"", ""인천"", ""광주"", ""대전"", ""울산"", ""세종"", ""경기"", ""강원"", ""충북"", ""충남"", ""전북"", ""전남"", ""경북"", ""경남"", ""제주""]","2016–2024 (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)",0
5,가족실태조사_가사노동_공동_참여도_2020_2023.csv,df15b5e3c958dbdca81dbf9bcf38d533956fc7d2258d8f717792b294acdb4865,898,2026-07-27T13:39:20.611000061+09:00,18,4,"[""지역"", ""세부지표"", ""2020"", ""2023""]","{""지역"": ""str"", ""세부지표"": ""str"", ""2020"": ""float64"", ""2023"": ""float64""}",household_chore_equal_pa

## 4. 28 × 18 × 9 완전 격자와 관측상태

In [5]:
grid = pd.MultiIndex.from_product(
    [REGIONS, indicator_ids, YEARS],
    names=['지역', '지표_id', '연도'],
).to_frame(index=False)

panel_work = grid.merge(
    input_long[['지역', '지표_id', '연도', '측정값', '원본행존재']],
    on=['지역', '지표_id', '연도'],
    how='left',
    validate='one_to_one',
)
panel_work['원본행존재'] = panel_work['원본행존재'].fillna(False).astype(bool)
panel_work['관측상태'] = np.where(panel_work['측정값'].notna(), '관측', '미관측')
panel_work = panel_work.merge(metadata, on='지표_id', how='left', validate='many_to_one')
panel_work['출처'] = panel_work['지표_id'].map(final_source_by_id)

panel_columns = [
    '지역', '지표_id', '지표명', '연도', '측정값', '단위', '출처', '원본행존재', '관측상태',
    '대분류', '세부영역', '방향',
]
panel = panel_work[panel_columns].copy()

assert len(panel) == 4536
assert panel['지표_id'].nunique() == 28
assert panel['지역'].nunique() == 18 and set(panel['지역']) == set(REGIONS)
assert set(panel['연도']) == set(YEARS)
assert not panel.duplicated(['지역', '연도', '지표_id']).any()
assert panel['단위'].notna().all() and panel['단위'].astype(str).str.strip().ne('').all()
assert panel['출처'].notna().all() and panel['출처'].astype(str).str.strip().ne('').all()
assert panel['원본행존재'].sum() == 4140
assert (~panel['원본행존재']).sum() == 396
assert panel['측정값'].notna().sum() == 3266
assert (panel['원본행존재'] & panel['측정값'].isna()).sum() == 874
assert panel['측정값'].isna().sum() == 1270
assert panel['관측상태'].value_counts().to_dict() == {'관측': 3266, '미관측': 1270}

print('완전 패널:', panel.shape)
print('원본행존재 / 신규 조합:', panel['원본행존재'].sum(), '/', (~panel['원본행존재']).sum())
display(panel.head(10))
display(panel['관측상태'].value_counts().rename_axis('관측상태').reset_index(name='행 수'))

완전 패널: (4536, 12)
원본행존재 / 신규 조합: 4140 / 396


,지역,지표_id,지표명,연도,측정값,단위,출처,원본행존재,관측상태,대분류,세부영역,방향
0,전국,youth_employment_rate,청년고용률,2016,62.240325,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
1,전국,youth_employment_rate,청년고용률,2017,61.770763,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
2,전국,youth_employment_rate,청년고용률,2018,62.037279,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
3,전국,youth_employment_rate,청년고용률,2019,62.596641,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
4,전국,youth_employment_rate,청년고용률,2020,61.253090,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
5,전국,youth_employment_rate,청년고용률,2021,63.107882,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
6,전국,youth_employment_rate,청년고용률,2022,66.382877,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
7,전국,youth_employment_rate,청년고용률,2023,67.627146,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
8,전국,youth_employment_rate,청년고용률,2024,68.427030,%,2016-2025_행정구역_시도__연령별_취업자_20260715184807.csv; 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv,True,관측,경제·고용·주거,고용여건,positive
9,전국,work_hours,근로시간,2016,NaN,시간/월(상용근로자 1인당),2016-2019_행정구역_시도___산업_규모별_임금_및_근로시간_상용근로자__상용근로자_1인이상_사업체__20260715183802.csv; 2020-2025_행정구역_시도___산업_규모별_임금_및_근로시간_상용근로자__상용근로자_1인이상_사업체__20260715184101.csv,False,미관측,사회·문화,일·가정 양립 여건,negative


,관측상태,행 수
0,관측,3266
1,미관측,1270


## 5. 저장·재로딩·키 기준 입력값 보존 검증

In [6]:
PANEL_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

panel.to_csv(PANEL_PATH, index=False, encoding='utf-8-sig')
input_file_qa.to_csv(INPUT_QA_PATH, index=False, encoding='utf-8-sig')
source_lineage.to_csv(SOURCE_QA_PATH, index=False, encoding='utf-8-sig')

reloaded = pd.read_csv(PANEL_PATH, encoding='utf-8-sig')
assert len(reloaded) == 4536
assert reloaded.columns.tolist() == panel_columns
assert not reloaded.duplicated(['지역', '연도', '지표_id']).any()
assert reloaded['지표_id'].nunique() == 28
assert reloaded['지역'].nunique() == 18 and set(reloaded['지역']) == set(REGIONS)
assert set(reloaded['연도']) == set(YEARS)
assert reloaded['단위'].notna().all() and reloaded['단위'].astype(str).str.strip().ne('').all()
assert reloaded['출처'].notna().all() and reloaded['출처'].astype(str).str.strip().ne('').all()
assert reloaded['원본행존재'].dtype == bool

input_observed = input_long.loc[input_long['측정값'].notna(), ['지역', '지표_id', '연도', '측정값', '원본파일', '원본행번호']].copy()
input_observed = input_observed.rename(columns={'측정값': '입력값'})
reloaded_values = reloaded[['지역', '지표_id', '연도', '측정값']].rename(columns={'측정값': '최종값'})
comparison = input_observed.merge(
    reloaded_values, on=['지역', '지표_id', '연도'], how='left', indicator=True, validate='one_to_one'
)
missing_input_observations = int(comparison['_merge'].ne('both').sum() + comparison['최종값'].isna().sum())
comparison['절대차'] = (comparison['입력값'] - comparison['최종값']).abs()
comparison['허용오차'] = VALUE_ATOL
comparison['결과'] = np.where(
    np.isclose(comparison['입력값'], comparison['최종값'], rtol=0.0, atol=VALUE_ATOL, equal_nan=False),
    'PASS', 'FAIL',
)
changed_values = int(comparison['결과'].eq('FAIL').sum())
input_keys = input_observed[['지역', '지표_id', '연도']].drop_duplicates()
output_observed_keys = reloaded.loc[reloaded['측정값'].notna(), ['지역', '지표_id', '연도']].drop_duplicates()
added_numeric_observations = int(output_observed_keys.merge(
    input_keys, on=['지역', '지표_id', '연도'], how='left', indicator=True
)['_merge'].eq('left_only').sum())
max_absolute_difference = float(comparison['절대차'].max())

assert len(comparison) == 3266
assert missing_input_observations == 0
assert added_numeric_observations == 0
assert changed_values == 0

value_preservation_qa = comparison[[
    '지역', '지표_id', '연도', '원본파일', '원본행번호', '입력값', '최종값', '절대차', '허용오차', '결과'
]].sort_values(['지표_id', '지역', '연도'], kind='stable').reset_index(drop=True)
value_preservation_qa.to_csv(VALUE_QA_PATH, index=False, encoding='utf-8-sig')

print('CSV 재로딩 자료형:')
print(reloaded.dtypes.to_string())
print('입력 관측 누락 / 추가 숫자 관측 / 값 변경:', missing_input_observations, '/', added_numeric_observations, '/', changed_values)
print('비교 허용오차(rtol, atol):', 0.0, VALUE_ATOL)
print('최대 절대차:', max_absolute_difference)
display(value_preservation_qa.head(10))

CSV 재로딩 자료형:
지역           str
지표_id        str
지표명          str
연도         int64
측정값      float64
단위           str
출처           str
원본행존재       bool
관측상태         str
대분류          str
세부영역         str
방향           str
입력 관측 누락 / 추가 숫자 관측 / 값 변경: 0 / 0 / 0
비교 허용오차(rtol, atol): 0.0 1e-12
최대 절대차: 2.842170943040401e-14


,지역,지표_id,연도,원본파일,원본행번호,입력값,최종값,절대차,허용오차,결과
0,강원,after_school_care_supply,2016,구조환경지표_21개_검증본.csv,83,1.828492,1.828492,0.0,1.000000e-12,PASS
1,강원,after_school_care_supply,2017,구조환경지표_21개_검증본.csv,83,1.854264,1.854264,0.0,1.000000e-12,PASS
2,강원,after_school_care_supply,2018,구조환경지표_21개_검증본.csv,83,1.897261,1.897261,0.0,1.000000e-12,PASS
3,강원,after_school_care_supply,2019,구조환경지표_21개_검증본.csv,83,2.062292,2.062292,0.0,1.000000e-12,PASS
4,강원,after_school_care_supply,2020,구조환경지표_21개_검증본.csv,83,2.184255,2.184255,0.0,1.000000e-12,PASS
5,강원,after_school_care_supply,2021,구조환경지표_21개_검증본.csv,83,2.301935,2.301935,0.0,1.000000e-12,PASS
6,강원,after_school_care_supply,2022,구조환경지표_21개_검증본.csv,83,2.434292,2.434292,0.0,1.000000e-12,PASS
7,강원,after_school_care_supply,2023,구조환경지표_21개_검증본.csv,83,2.558583,2.558583,0.0,1.000000e-12,PASS
8,강원,after_school_care_supply,2024,구조환경지표_21개_검증본.csv,83,2.837412,2.837412,0.0,1.000000e-12,PASS
9,경기,after_school_care_supply,2016,구조환경지표_21개_검증본.csv,82,0.854457,0.854457,0.0,1.000000e-12,PASS


## 6. 필수 QA 기대값 대조와 보고서 생성

In [7]:
manifest_missing = len(set(indicator_ids) - set(reloaded['지표_id']))
manifest_extra = len(set(reloaded['지표_id']) - set(indicator_ids))
invalid_regions_or_years = int((~reloaded['지역'].isin(REGIONS)).sum() + (~reloaded['연도'].isin(YEARS)).sum())
unit_missing = int(reloaded['단위'].isna().sum() + reloaded['단위'].astype('string').str.strip().eq('').sum())
source_missing = int(reloaded['출처'].isna().sum() + reloaded['출처'].astype('string').str.strip().eq('').sum())
empty_source_indicators = sum(not str(final_source_by_id.get(indicator_id, '')).strip() for indicator_id in indicator_ids)

qa_items = [
    ('최종 패널 행 수', '4,536', f"{len(reloaded):,}"),
    ('지표 ID 수', '28', str(reloaded['지표_id'].nunique())),
    ('지역 수', '18', str(reloaded['지역'].nunique())),
    ('연도', '2016–2024', f"{reloaded['연도'].min()}–{reloaded['연도'].max()}"),
    ('지역×연도×지표 중복 키', '0', str(reloaded.duplicated(['지역', '연도', '지표_id']).sum())),
    ('매니페스트 대비 지표 누락·초과', '0', str(manifest_missing + manifest_extra)),
    ('허용되지 않은 지역·연도', '0', str(invalid_regions_or_years)),
    ('단위 결측', '0/4,536', f"{unit_missing}/{len(reloaded):,}"),
    ('출처 결측', '0/4,536', f"{source_missing}/{len(reloaded):,}"),
    ('입력 숫자 관측값 변경', '0', str(changed_values)),
    ('원본 존재 조합', '4,140', f"{int(reloaded['원본행존재'].sum()):,}"),
    ('완전 격자 신규 조합', '396', f"{int((~reloaded['원본행존재']).sum()):,}"),
    ('숫자 관측', '3,266', f"{int(reloaded['측정값'].notna().sum()):,}"),
    ('원본 셀 결측', '874', f"{int((reloaded['원본행존재'] & reloaded['측정값'].isna()).sum()):,}"),
    ('최종 미관측', '1,270', f"{int(reloaded['측정값'].isna().sum()):,}"),
    ('관측+미관측', '3,266+1,270=4,536', f"{int(reloaded['측정값'].notna().sum()):,}+{int(reloaded['측정값'].isna().sum()):,}={len(reloaded):,}"),
    ('raw_sources 원본 항목', '53', str(len(source_lineage))),
    ('comparison_series 제외', '1', str(comparison_excluded)),
    ('최종 출처 구성요소', '52', str(included_source_components)),
    ('출처가 비는 지표', '0', str(empty_source_indicators)),
    ('누락된 입력 관측값', '0', str(missing_input_observations)),
    ('추가 생성된 숫자 관측값', '0', str(added_numeric_observations)),
]
panel_qa = pd.DataFrame(qa_items, columns=['검증 항목', '기대값', '실제값'])
panel_qa['결과'] = np.where(panel_qa['기대값'].eq(panel_qa['실제값']), 'PASS', 'FAIL')
panel_qa['비고'] = ''
panel_qa.loc[panel_qa['검증 항목'].eq('입력 숫자 관측값 변경'), '비고'] = f'rtol=0, atol={VALUE_ATOL:g}; 반올림·원본값 변경 없음; 최대 절대차={max_absolute_difference:g}'
assert panel_qa['결과'].eq('PASS').all(), panel_qa.loc[panel_qa['결과'].eq('FAIL')].to_dict('records')
panel_qa.to_csv(PANEL_QA_PATH, index=False, encoding='utf-8-sig')

def markdown_table(frame: pd.DataFrame) -> str:
    safe = frame.astype(str).apply(lambda column: column.str.replace('|', r'\|', regex=False).str.replace('\n', ' ', regex=False))
    header = '| ' + ' | '.join(safe.columns) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(safe.columns)) + ' |'
    rows = ['| ' + ' | '.join(row) + ' |' for row in safe.itertuples(index=False, name=None)]
    return '\n'.join([header, separator, *rows])

mapping_report = input_file_qa[['파일명', '대응 지표', '행 수', '열 수', '중복 키 수']].copy()
report = f"""# 구조환경지표 28개 보간 전 통합패널 QA

- 실행일: {pd.Timestamp.now(tz='Asia/Seoul').isoformat()}
- 최종 판정: **GO**
- 입력: `data/interim/`의 승인된 정확한 파일명 8개만 사용
- 처리 제외: 보간, 결측 대체, 정규화, 표준화, 방향 변환, 가중치, 구조환경지수 산출
- 경고: 클린 커널 전체 실행에서 결과에 영향을 주는 경고 없음

## 산출물

- `{PANEL_PATH.relative_to(ROOT).as_posix()}`
- `{INPUT_QA_PATH.relative_to(ROOT).as_posix()}`
- `{PANEL_QA_PATH.relative_to(ROOT).as_posix()}`
- `{SOURCE_QA_PATH.relative_to(ROOT).as_posix()}`
- `{VALUE_QA_PATH.relative_to(ROOT).as_posix()}`

## 입력 8개와 지표 대응

{markdown_table(mapping_report)}

28개 지표가 매니페스트와 입력 사이에서 누락·초과·중복 없이 1:1 대응한다. 기존 21개 파일의 명칭 차이 5건은 명시적 사전으로 연결했다. 고용 CSV는 정확한 승인 파일명, `시도`+2016–2024 열, 표준 17개 시도를 검증한 뒤 `youth_regular_employment_rate`를 부여했다.

## 필수 QA

{markdown_table(panel_qa)}

## 입력값 보존

- 누락된 입력 숫자 관측: {missing_input_observations}건
- 추가 생성된 숫자 관측: {added_numeric_observations}건
- 값이 달라진 관측: {changed_values}건
- 비교: `rtol=0`, `atol={VALUE_ATOL:g}`; 반올림과 원본값 변경 없음
- 최대 절대차: {max_absolute_difference:g}

## 단위·출처와 계보

- 단위 결측: {unit_missing}/{len(reloaded):,}
- 출처 결측: {source_missing}/{len(reloaded):,}
- raw_sources 원본 항목: {len(source_lineage)}건
- `housing_price` comparison series 제외: {comparison_excluded}건 (`QA comparison series`)
- 최종 출처 구성요소: {included_source_components}건
- 원본 `raw_sources`는 순서와 중첩 구조를 `raw_source_json`에 보존

## 재로딩 검증

저장한 패널 CSV를 다시 읽어 열, 자료형, 4,536행, 28개 지표, 18개 지역, 2016–2024년, 중복 키 0건, 단위·출처 결측 0건과 입력값 보존을 재검증했다. `측정값`은 `float64`, `연도`는 정수형, `원본행존재`는 불리언으로 재로딩됐다.
"""
REPORT_PATH.write_text(report, encoding='utf-8')

display(panel_qa)
display(Markdown(f"**최종 판정: GO** — {len(reloaded):,}행, QA {panel_qa['결과'].eq('PASS').sum()}/{len(panel_qa)} PASS"))
print('저장 완료:')
for path in [PANEL_PATH, INPUT_QA_PATH, PANEL_QA_PATH, SOURCE_QA_PATH, VALUE_QA_PATH, REPORT_PATH]:
    print(' -', path.relative_to(ROOT))

,검증 항목,기대값,실제값,결과,비고
0,최종 패널 행 수,"4,536","4,536",PASS,
1,지표 ID 수,28,28,PASS,
2,지역 수,18,18,PASS,
3,연도,2016–2024,2016–2024,PASS,
4,지역×연도×지표 중복 키,0,0,PASS,
5,매니페스트 대비 지표 누락·초과,0,0,PASS,
6,허용되지 않은 지역·연도,0,0,PASS,
7,단위 결측,"0/4,536","0/4,536",PASS,
8,출처 결측,"0/4,536","0/4,536",PASS,
9,입력 숫자 관측값 변경,0,0,PASS,"rtol=0, atol=1e-12; 반올림·원본값 변경 없음; 최대 절대차=2.84217e-14"


**최종 판정: GO** — 4,536행, QA 22/22 PASS

저장 완료:
 - data\processed\구조환경지표_28개_보간전_기준패널.csv
 - reports\20260804_구조환경지표_28개_입력파일_QA.csv
 - reports\20260804_구조환경지표_28개_패널완전성_관측상태_QA.csv
 - reports\20260804_구조환경지표_28개_raw_sources_출처계보_QA.csv
 - reports\20260804_구조환경지표_28개_입력값보존_QA.csv
 - reports\20260804_구조환경지표_28개_보간전_통합패널_QA.md
